In [ ]:
import os
import cv2
import argparse
from ultralytics  import YOLO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from analyze_errors.matching import match_gt_with_results
from analyze_errors.metrics_calculate import compute_map, box_metrics_calculate
from analyze_errors.visualization import visualizete_predict


# === parameters ===
RANDOM_STATE = 42

In [ ]:
path2data_file = "./data.yaml"
split = "val"
path2model = "./runs/detect/baseline_v0/aug_none_only_detect/weights/best.pt"
imgsize = 512
project = "baseline_v0"
exp_name = "errors_analyze"
conf = 0.0001

nms_iou = 0.6
match_iou = 0.5

batch_size = 16

In [ ]:
# === model init ===
model = YOLO(path2model)

In [ ]:
results = model.predict(
    source="./dataset_yolo/val_fold_1.txt",
    imgsz=imgsize,
    batch=batch_size,
    workers=2,
    seed=RANDOM_STATE,
    deterministic=True,
    conf=conf, # 0.0001 # низкий порог для честного расчёта mAP
    iou=nms_iou, # IoU для NMS при валидации
    verbose=False # Отключить печать
)

### mAP calculate

In [ ]:
m = compute_map(results, "./dataset_yolo/labels", num_classes=1)
print(f"mAP0.5: {m['map_05']:0.4f}")
print(f"mAP0.5-0.95: {m['map_50_95']:.4f}")
print(m)

### TP, FP, FN, Precision, Recall for GT

In [ ]:
boxes_metrics = []
for c in np.linspace(0.05, 0.9, 18):
    m = box_metrics_calculate(results, "./dataset_yolo/labels", iou=match_iou, conf=c)
    boxes_metrics.append({
        "conf": c,
        "iou": match_iou,
        "metrics": m 
    })

### Visualization

In [ ]:
result_item = results[10]
stem = result_item.path.split("\\")[-1].split('.')[0]
path2label = f"./dataset_yolo/labels/{stem}.txt"

matches, un_gt, un_pred, gt = match_gt_with_results(
    result_item, path2label, iou_thresh=match_iou, conf_thresh=conf, return_gt=True
)

In [ ]:
visualizete_predict(
    result_item, matches, un_gt, un_pred, gt
)